# FOXF1_BMP4_timelapse — 03d_segmentation_refinement_ilastik

**Feeds:** Fig 5i, ED Fig 10f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# FOXF1/BMP4 Timelapse: Step 3d Segmentation Refinement From Ilastik Probabilities

This notebook is a clean segmentation-only branch from the `03c` ilastik workflow.

The goal is **not** perfect nuclear outlines. The goal is a segmentation that:

- contains nuclei reasonably well
- stays single-component
- avoids obvious blebs / debris
- avoids half-moon or concave collapsed masks
- gives stable centroids for downstream tracking and measurement

We keep tracking out of this notebook on purpose. First we want to see the segmentation clearly,
understand the current failure modes, and test one or two higher-leverage refinements.


In [ ]:
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

from IPython.display import display
from scipy import ndimage as ndi
from skimage import color, feature, filters, measure, morphology, segmentation

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
np.set_printoptions(suppress=True, precision=4)


In [ ]:
# ----------------------------- #
# Configuration
# ----------------------------- #
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if (cand / "ilastik" / "datasets" / "20260213" / "batch_processing_data_v0").exists() and (cand / "scripts").exists():
            return cand
    return start


ROOT = find_project_root(Path.cwd())
ILASTIK_DIR = ROOT / "ilastik" / "datasets" / "20260213" / "batch_processing_data_v0"
RAW_PATH = ILASTIK_DIR / "1-Pos009_012.tif"
PROB_PATH = ILASTIK_DIR / "1-Pos009_012_Probabilities.tiff"
BASELINE_PARAMS_PATH = (
    ROOT
    / "results"
    / "datasets"
    / "20260213"
    / "segmentation_03c_ilastik"
    / "search"
    / "stage2_candidate_16_splitcontrol"
    / "best_params.json"
)

RESULTS_DIR = ROOT / "results" / "datasets" / "20260213" / "segmentation_03d_ilastik_refinement"
QUICK_DIR = RESULTS_DIR / "quick_checks"
GALLERY_DIR = RESULTS_DIR / "failure_gallery"
TABLE_DIR = RESULTS_DIR / "tables"
for _d in [RESULTS_DIR, QUICK_DIR, GALLERY_DIR, TABLE_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

POSITION_LABEL = "1-Pos009_012"
MIN_PER_FRAME = 10.0

STEP_DIAG_TIMES = None
CONSECUTIVE_AUDIT_HALF_WINDOW = 2
PREVIEW_TIMES = [0, 20, 40, 80, 120, 160, 220, 280, 299]
FAILURE_GALLERY_N = 8
GOOD_GALLERY_N = 8
FAILURE_CROP_HALF_SIZE = 34
RUN_FULL_BASELINE_SCAN = True
RUN_REFINEMENT_SCAN = True

REFINE_CFG = {
    "compactify_enabled": True,
    "compactify_min_area": 95,
    "compactify_low_solidity": 0.82,
    "compactify_low_circularity": 0.72,
    "compactify_close_radius": 1,
    "compactify_fill_hole_area": 96,
    "compactify_use_convex_hull": True,
    "compactify_max_area_gain_frac": 0.35,
    "compactify_max_expand_radius": 6,
    "compactify_keep_if_centroid_shift_lt": 4.0,
}

print("ROOT:", ROOT)
print("RESULTS_DIR:", RESULTS_DIR)


## 1) Load Data And Reconfirm The Ilastik Export Layout

This is the same raw/probability pair we used in `03c`, but this notebook starts fresh:

- raw stack: `(BF, RFP)` per timepoint
- probability stack: `(p_nuc, p_non_nuc)` per timepoint
- downstream segmentation only uses `p_nuc`


In [ ]:
assert RAW_PATH.exists(), f"Missing raw input: {RAW_PATH}"
assert PROB_PATH.exists(), f"Missing probability input: {PROB_PATH}"
assert BASELINE_PARAMS_PATH.exists(), f"Missing baseline params: {BASELINE_PARAMS_PATH}"

with tifffile.TiffFile(RAW_PATH) as tf_raw:
    n_pages_raw = len(tf_raw.pages)
    shape_raw = tf_raw.pages[0].shape
    dtype_raw = tf_raw.pages[0].dtype

with tifffile.TiffFile(PROB_PATH) as tf_prob:
    n_pages_prob = len(tf_prob.pages)
    shape_prob = tf_prob.pages[0].shape
    dtype_prob = tf_prob.pages[0].dtype

raw_mm = tifffile.memmap(RAW_PATH)
prob_mm = tifffile.memmap(PROB_PATH)

print(f"RAW: pages={n_pages_raw}, shape={shape_raw}, dtype={dtype_raw}, memmap={raw_mm.shape}")
print(f"PROB: pages={n_pages_prob}, shape={shape_prob}, dtype={dtype_prob}, memmap={prob_mm.shape}")

assert raw_mm.ndim == 4 and raw_mm.shape[1] == 2, "Expected raw memmap shape (T,2,Y,X)."
assert prob_mm.ndim == 4 and prob_mm.shape[1] == 2, "Expected prob memmap shape (T,2,Y,X)."

N_TIME = int(raw_mm.shape[0])
print(f"Derived timepoints: {N_TIME}")

sample_times = [0, 1, 20, 50, 100, 150, 200, 250, N_TIME - 1]
sample_times = sorted({i for i in sample_times if 0 <= i < N_TIME})

pair_checks = []
for t in sample_times:
    p0 = prob_mm[t, 0].astype(np.float32, copy=False)
    p1 = prob_mm[t, 1].astype(np.float32, copy=False)
    s = p0 + p1
    pair_checks.append(
        {
            "t": int(t),
            "p_nuc_mean": float(p0.mean()),
            "p_non_nuc_mean": float(p1.mean()),
            "sum_mean": float(s.mean()),
            "sum_std": float(s.std()),
            "sum_min": float(s.min()),
            "sum_max": float(s.max()),
            "max_abs_err_from_1": float(np.max(np.abs(s - 1.0))),
        }
    )

pair_checks_df = pd.DataFrame(pair_checks)
display(pair_checks_df)


## 2) Segmentation Scope For 03d

This notebook is now strictly segmentation-focused.

We are **not** doing illumination QC here and we are **not** excluding the top-left corner during segmentation.
The segmentation branch should be judged on the ilastik probability map and the resulting masks themselves.

Any spatial reliability masking or illumination correction can be reintroduced later during fluorescence
quantification, where those decisions affect measurement rather than segmentation.


In [ ]:
valid_mask = np.ones(shape_raw, dtype=bool)
print("Segmentation valid-mask mode: full field")
print(f"Valid pixels: {valid_mask.mean() * 100:.2f}%")


## 3) Load The Current 03c Baseline And Define A Refinement-Friendly Segmenter

The baseline parameters below are the accepted `03c` starting point. We keep the same probability-first
segmentation logic, but add one optional post-segmentation compactification pass so we can test whether
the half-moon / concave masks can be repaired without needing to retrain ilastik immediately.


In [ ]:
with open(BASELINE_PARAMS_PATH) as f:
    BASELINE_SEG_PARAMS = json.load(f)

print("Baseline segmentation params loaded from:")
print(BASELINE_PARAMS_PATH)
display(pd.Series(BASELINE_SEG_PARAMS, name="baseline_param"))


def clamp_probs(p: np.ndarray) -> np.ndarray:
    return np.clip(p.astype(np.float32, copy=False), 0.0, 1.0)


def region_stats_from_labels(labels: np.ndarray, intensity_image: np.ndarray | None = None) -> pd.DataFrame:
    if int(labels.max()) == 0:
        return pd.DataFrame(
            columns=[
                "label",
                "area",
                "solidity",
                "eccentricity",
                "perimeter",
                "circularity",
                "mean_intensity",
                "max_intensity",
                "centroid-0",
                "centroid-1",
                "bbox-0",
                "bbox-1",
                "bbox-2",
                "bbox-3",
            ]
        )
    props = measure.regionprops_table(
        labels,
        intensity_image=intensity_image,
        properties=(
            "label",
            "area",
            "solidity",
            "eccentricity",
            "perimeter",
            "centroid",
            "bbox",
            "mean_intensity",
            "max_intensity",
        ),
    )
    df = pd.DataFrame(props)
    per = df["perimeter"].to_numpy(dtype=float)
    area = df["area"].to_numpy(dtype=float)
    circ = np.where(per > 0, 4.0 * np.pi * area / (per ** 2), np.nan)
    df["circularity"] = circ
    return df


def compactify_labels(labels: np.ndarray, p_sm: np.ndarray, cfg: dict | None = None) -> np.ndarray:
    if not cfg or not bool(cfg.get("compactify_enabled", False)) or int(labels.max()) == 0:
        return labels.astype(np.int32, copy=True)

    out = np.zeros_like(labels, dtype=np.int32)
    next_label = 1
    props = measure.regionprops(labels, intensity_image=p_sm)

    for r in props:
        mask = labels == int(r.label)
        area = float(r.area)
        solidity = float(r.solidity) if r.solidity is not None else np.nan
        eccentricity = float(r.eccentricity) if r.eccentricity is not None else np.nan
        perimeter = float(r.perimeter) if r.perimeter is not None else np.nan
        circularity = (4.0 * np.pi * area / (perimeter ** 2)) if (np.isfinite(perimeter) and perimeter > 0) else np.nan

        use_repair = (
            area >= float(cfg["compactify_min_area"])
            and (
                (np.isfinite(solidity) and solidity < float(cfg["compactify_low_solidity"]))
                or (np.isfinite(circularity) and circularity < float(cfg["compactify_low_circularity"]))
            )
        )

        repaired = mask.copy()
        if use_repair:
            if int(cfg["compactify_close_radius"]) > 0:
                repaired = morphology.binary_closing(
                    repaired, footprint=morphology.disk(int(cfg["compactify_close_radius"]))
                )
            repaired = morphology.remove_small_holes(
                repaired, area_threshold=int(cfg["compactify_fill_hole_area"])
            )

            if bool(cfg.get("compactify_use_convex_hull", True)):
                hull = morphology.convex_hull_image(repaired)
                envelope = morphology.binary_dilation(
                    mask, footprint=morphology.disk(int(cfg["compactify_max_expand_radius"]))
                )
                candidate = hull & envelope
                if candidate.sum() > 0:
                    repaired = candidate

            max_area = area * (1.0 + float(cfg["compactify_max_area_gain_frac"]))
            if repaired.sum() > max_area:
                repaired = mask.copy()
            else:
                old_cy, old_cx = r.centroid
                yx = np.argwhere(repaired)
                if yx.size > 0:
                    new_cy, new_cx = yx.mean(axis=0)
                    shift = float(np.hypot(new_cy - old_cy, new_cx - old_cx))
                    if shift > float(cfg["compactify_keep_if_centroid_shift_lt"]):
                        repaired = mask.copy()

        target = repaired if repaired.any() else mask
        target = target & (out == 0)
        if not target.any():
            target = mask & (out == 0)
        if not target.any():
            continue
        out[target] = next_label
        next_label += 1

    out, _, _ = segmentation.relabel_sequential(out)
    return out.astype(np.int32)


def segment_probability_frame(
    p_nuc: np.ndarray,
    valid_mask: np.ndarray,
    params: dict,
    compact_cfg: dict | None = None,
    return_debug: bool = False,
) -> dict:
    p = clamp_probs(p_nuc)
    p_sm = filters.gaussian(p, sigma=float(params["smooth_sigma"]), preserve_range=True)
    p_sm = np.where(valid_mask, p_sm, 0.0)
    debug_steps = {}
    if return_debug:
        debug_steps["p_raw"] = p.copy()
        debug_steps["p_smooth"] = p_sm.copy()

    low = (p_sm >= float(params["low_prob_thr"])) & valid_mask
    high = (p_sm >= float(params["high_prob_thr"])) & valid_mask
    hys = ndi.binary_propagation(high, mask=low)

    if return_debug:
        debug_steps["mask_low"] = low.copy()
        debug_steps["mask_high"] = high.copy()
        debug_steps["hys_pre_fallback"] = hys.copy()

    if not np.any(hys):
        hys = low.copy()
    if return_debug:
        debug_steps["hys_after_fallback"] = hys.copy()

    if int(params["open_radius"]) > 0:
        hys = morphology.binary_opening(hys, footprint=morphology.disk(int(params["open_radius"])))
    if int(params["close_radius"]) > 0:
        hys = morphology.binary_closing(hys, footprint=morphology.disk(int(params["close_radius"])))
    hys = morphology.remove_small_holes(hys, area_threshold=int(params["min_hole_area"]))
    hys = morphology.remove_small_objects(hys, min_size=max(8, int(params["min_area"] // 2)))
    hys = hys & valid_mask
    if return_debug:
        debug_steps["hys_post_morph"] = hys.copy()

    peak_coords = feature.peak_local_max(
        p_sm,
        labels=hys.astype(np.uint8),
        min_distance=int(params["seed_min_distance"]),
        threshold_abs=float(params["seed_prob_thr"]),
        num_peaks=int(params["max_seed_peaks"]),
        exclude_border=False,
    )
    markers = np.zeros_like(hys, dtype=np.int32)
    if peak_coords.size > 0:
        markers[peak_coords[:, 0], peak_coords[:, 1]] = np.arange(1, peak_coords.shape[0] + 1, dtype=np.int32)
        markers = ndi.label(markers > 0)[0].astype(np.int32)
    if return_debug:
        debug_steps["markers_peak"] = markers.copy()

    if markers.max() == 0:
        markers = ndi.label(hys)[0].astype(np.int32)
    else:
        cc = ndi.label(hys)[0]
        for cc_id in range(1, int(cc.max()) + 1):
            m = cc == cc_id
            if not np.any(markers[m] > 0):
                rr, ccx = np.where(m)
                if rr.size > 0:
                    idx = np.argmax(p_sm[rr, ccx])
                    markers[rr[idx], ccx[idx]] = markers.max() + 1
        markers = ndi.label(markers > 0)[0].astype(np.int32)
    if return_debug:
        debug_steps["markers_final"] = markers.copy()

    labels = segmentation.watershed(-p_sm, markers=markers, mask=hys).astype(np.int32)
    if return_debug:
        debug_steps["labels_watershed"] = labels.copy()

    def _shape_stats(lbl_arr: np.ndarray) -> pd.DataFrame:
        return region_stats_from_labels(lbl_arr, intensity_image=p_sm)

    def _adjacency_boundary_means(lbl_arr: np.ndarray) -> dict:
        if int(lbl_arr.max()) <= 1:
            return {}
        base = int(lbl_arr.max()) + 1
        pid_parts = []
        val_parts = []
        for la, lb, va, vb in [
            (lbl_arr[:, :-1], lbl_arr[:, 1:], p_sm[:, :-1], p_sm[:, 1:]),
            (lbl_arr[:-1, :], lbl_arr[1:, :], p_sm[:-1, :], p_sm[1:, :]),
        ]:
            m = (la > 0) & (lb > 0) & (la != lb)
            if not np.any(m):
                continue
            a = la[m].astype(np.int32)
            b = lb[m].astype(np.int32)
            lo = np.minimum(a, b)
            hi = np.maximum(a, b)
            pid = lo.astype(np.int64) * base + hi.astype(np.int64)
            vv = 0.5 * (va[m].astype(np.float32) + vb[m].astype(np.float32))
            pid_parts.append(pid)
            val_parts.append(vv)
        if not pid_parts:
            return {}
        pid_all = np.concatenate(pid_parts, axis=0)
        val_all = np.concatenate(val_parts, axis=0).astype(np.float64)
        uniq, inv = np.unique(pid_all, return_inverse=True)
        sums = np.bincount(inv, weights=val_all)
        cnts = np.bincount(inv)
        out = {}
        for u, s, c in zip(uniq, sums, cnts):
            a = int(u // base)
            b = int(u % base)
            out[(a, b)] = {"boundary_mean_prob": float(s / max(c, 1)), "contact_px": int(c)}
        return out

    if bool(params.get("split_merge_enabled", True)) and int(labels.max()) > 1:
        max_passes = int(params.get("split_merge_max_passes", 0))
        for _ in range(max_passes):
            stats_df = _shape_stats(labels)
            if stats_df.empty or int(labels.max()) <= 1:
                break
            stats_df = stats_df.set_index("label")
            pair_info = _adjacency_boundary_means(labels)
            if not pair_info:
                break
            best_pair = None
            best_priority = -np.inf
            for (a, b), info in pair_info.items():
                if (a not in stats_df.index) or (b not in stats_df.index):
                    continue
                contact_px = int(info["contact_px"])
                if contact_px < int(params["split_merge_min_contact_px"]):
                    continue
                boundary_mean = float(info["boundary_mean_prob"])
                if boundary_mean < float(params["split_merge_boundary_prob_thr"]):
                    continue
                ra = stats_df.loc[a]
                rb = stats_df.loc[b]
                dist = float(
                    np.hypot(
                        float(ra["centroid-0"]) - float(rb["centroid-0"]),
                        float(ra["centroid-1"]) - float(rb["centroid-1"]),
                    )
                )
                if dist > float(params["split_merge_max_centroid_dist"]):
                    continue
                area_a = float(ra["area"])
                area_b = float(rb["area"])
                area_ratio = max(area_a, area_b) / max(min(area_a, area_b), 1e-6)

                merged_mask = (labels == int(a)) | (labels == int(b))
                mprops = measure.regionprops(merged_mask.astype(np.uint8))
                if len(mprops) == 0:
                    continue
                mr = mprops[0]
                m_sol = float(mr.solidity) if mr.solidity is not None else np.nan
                m_per = float(mr.perimeter) if mr.perimeter is not None else np.nan
                m_circ = (4.0 * np.pi * float(mr.area) / (m_per ** 2)) if (np.isfinite(m_per) and m_per > 0) else np.nan
                if (not np.isfinite(m_sol)) or (not np.isfinite(m_circ)):
                    continue

                sa = float(ra["solidity"]) if np.isfinite(ra["solidity"]) else 0.0
                sb = float(rb["solidity"]) if np.isfinite(rb["solidity"]) else 0.0
                ca = float(ra["circularity"]) if np.isfinite(ra["circularity"]) else 0.0
                cb = float(rb["circularity"]) if np.isfinite(rb["circularity"]) else 0.0
                before_shape = max(sa + ca, sb + cb)
                after_shape = m_sol + m_circ
                gain = float(after_shape - before_shape)

                small_cond = min(area_a, area_b) <= float(params["split_merge_small_area"])
                shape_cond = gain >= float(params["split_merge_shape_gain_min"])
                if (not small_cond) and (area_ratio < 1.7) and (
                    gain < float(params["split_merge_shape_gain_min"]) + 0.10
                ):
                    continue
                if not (small_cond or shape_cond):
                    continue

                priority = (
                    gain
                    + 0.35 * (boundary_mean - float(params["split_merge_boundary_prob_thr"]))
                    + 0.002 * float(contact_px)
                )
                if priority > best_priority:
                    best_priority = priority
                    best_pair = (int(a), int(b))
            if best_pair is None:
                break
            a, b = best_pair
            labels[labels == b] = a
            labels, _, _ = segmentation.relabel_sequential(labels)
    if return_debug:
        debug_steps["labels_splitmerge"] = labels.copy()

    if compact_cfg:
        labels = compactify_labels(labels, p_sm, compact_cfg)
    if return_debug:
        debug_steps["labels_compactified"] = labels.copy()

    props = measure.regionprops(labels, intensity_image=p_sm)
    keep = []
    rows = []
    for r in props:
        area = float(r.area)
        solidity = float(r.solidity) if r.solidity is not None else np.nan
        ecc = float(r.eccentricity) if r.eccentricity is not None else np.nan
        mean_p = float(r.mean_intensity)
        max_p = float(r.max_intensity)
        perimeter = float(r.perimeter) if r.perimeter is not None else np.nan
        circ = (4.0 * np.pi * area / (perimeter ** 2)) if (np.isfinite(perimeter) and perimeter > 0) else np.nan
        ok = True
        if area < float(params["min_area"]) or area > float(params["max_area"]):
            ok = False
        if np.isfinite(solidity) and solidity < float(params["min_solidity"]):
            ok = False
        if np.isfinite(ecc) and ecc > float(params["max_eccentricity"]):
            ok = False
        if mean_p < float(params["min_mean_prob"]):
            ok = False

        rows.append(
            {
                "label": int(r.label),
                "area": area,
                "solidity": solidity,
                "eccentricity": ecc,
                "mean_prob": mean_p,
                "max_prob": max_p,
                "circularity": circ,
                "centroid_y": float(r.centroid[0]),
                "centroid_x": float(r.centroid[1]),
                "bbox_min_row": int(r.bbox[0]),
                "bbox_min_col": int(r.bbox[1]),
                "bbox_max_row": int(r.bbox[2]),
                "bbox_max_col": int(r.bbox[3]),
                "keep": bool(ok),
            }
        )
        if ok:
            keep.append(int(r.label))

    if keep:
        labels = np.where(np.isin(labels, np.array(keep, dtype=np.int32)), labels, 0)
        labels, _, _ = segmentation.relabel_sequential(labels)
    else:
        labels = np.zeros_like(labels, dtype=np.int32)
    if return_debug:
        debug_steps["labels_final"] = labels.copy()

    kept = pd.DataFrame(rows)
    if not kept.empty:
        kept = kept[kept["keep"]].copy().reset_index(drop=True)

    out = {
        "p_smooth": p_sm.astype(np.float32),
        "binary": hys.astype(bool),
        "labels": labels.astype(np.int32),
        "markers": markers.astype(np.int32),
        "regions_df": kept,
        "peak_coords": peak_coords,
    }
    if return_debug:
        out["debug_steps"] = debug_steps
    return out


In [ ]:
def _robust_gray_display(
    gray: np.ndarray,
    low_pct: float = 1.0,
    high_pct: float = 99.5,
    mask: np.ndarray | None = None,
) -> np.ndarray:
    g = gray.astype(np.float32)
    vm = mask if (mask is not None and mask.shape == g.shape) else (valid_mask if valid_mask.shape == g.shape else None)
    vals = g[vm] if vm is not None else g.reshape(-1)
    if vals.size == 0:
        lo, hi = float(np.min(g)), float(np.max(g))
    else:
        lo = float(np.percentile(vals, low_pct))
        hi = float(np.percentile(vals, high_pct))
    if hi <= lo:
        hi = lo + 1.0
    out = np.clip((g - lo) / (hi - lo), 0.0, 1.0)
    out = out.copy()
    if vm is not None:
        out[~vm] = 0.0
    return out


def overlay_labels_on_gray(
    gray: np.ndarray,
    labels: np.ndarray,
    alpha: float = 0.45,
    mask: np.ndarray | None = None,
) -> np.ndarray:
    base = _robust_gray_display(gray, mask=mask)
    rgb = np.dstack([base, base, base])
    if labels.max() > 0:
        rgb = color.label2rgb(labels, image=rgb, alpha=alpha, bg_label=0, image_alpha=1.0)
    return np.clip(rgb, 0.0, 1.0)


def overlay_boundaries(
    gray: np.ndarray,
    labels: np.ndarray,
    color_rgb=(1.0, 0.2, 0.2),
    mask: np.ndarray | None = None,
) -> np.ndarray:
    base = _robust_gray_display(gray, mask=mask)
    rgb = np.dstack([base, base, base])
    ov = segmentation.mark_boundaries(rgb, labels > 0, color=color_rgb, mode="thick")
    return np.clip(ov, 0.0, 1.0)


def overlay_seed_points(
    gray: np.ndarray,
    peak_coords: np.ndarray,
    mask: np.ndarray | None = None,
    color_rgb=(1.0, 0.2, 0.2),
    radius_px: int = 2,
) -> np.ndarray:
    base = _robust_gray_display(gray, mask=mask)
    rgb = np.dstack([base, base, base])
    if peak_coords is None or len(peak_coords) == 0:
        return np.clip(rgb, 0.0, 1.0)
    yy, xx = np.indices(gray.shape)
    color_arr = np.array(color_rgb, dtype=np.float32)
    for y, x in np.asarray(peak_coords, dtype=int):
        disk = (yy - int(y)) ** 2 + (xx - int(x)) ** 2 <= int(radius_px) ** 2
        rgb[disk] = color_arr
    return np.clip(rgb, 0.0, 1.0)


def crop_slices(cy: float, cx: float, half_size: int, shape: tuple[int, int]) -> tuple[slice, slice]:
    h, w = shape
    y0 = max(0, int(round(cy)) - half_size)
    y1 = min(h, int(round(cy)) + half_size)
    x0 = max(0, int(round(cx)) - half_size)
    x1 = min(w, int(round(cx)) + half_size)
    return slice(y0, y1), slice(x0, x1)


def frame_quality_metrics(p_nuc: np.ndarray, labels: np.ndarray, valid_mask: np.ndarray, params: dict) -> dict:
    p = clamp_probs(p_nuc)
    v = valid_mask
    m = (labels > 0) & v

    p_mass_total = float(np.sum(p[v]))
    p_mass_in_mask = float(np.sum(p[m]))
    p_mass_capture = p_mass_in_mask / max(p_mass_total, 1e-8)

    high_thr = float(params["high_recall_prob_thr"])
    low_thr = float(params["low_contam_prob_thr"])
    hi = (p >= high_thr) & v
    lo = (p <= low_thr) & v

    high_recall = float(np.mean(m[hi])) if np.any(hi) else np.nan
    low_contam = float(np.mean(lo[m])) if np.any(m) else np.nan

    n_obj = int(labels.max())
    if n_obj > 0:
        props = measure.regionprops(labels)
        areas = np.array([r.area for r in props], dtype=float)
        solid = np.array([float(r.solidity) if r.solidity is not None else np.nan for r in props], dtype=float)
        ecc = np.array([float(r.eccentricity) if r.eccentricity is not None else np.nan for r in props], dtype=float)
        circ = []
        for r in props:
            per = float(r.perimeter) if r.perimeter is not None else np.nan
            circ.append(float(4.0 * np.pi * r.area / (per ** 2)) if (np.isfinite(per) and per > 0) else np.nan)
        circ = np.array(circ, dtype=float)

        area_med = float(np.nanmedian(areas))
        solidity_med = float(np.nanmedian(solid)) if np.any(np.isfinite(solid)) else np.nan
        circularity_med = float(np.nanmedian(circ)) if np.any(np.isfinite(circ)) else np.nan
        low_solidity_frac = float(np.nanmean(solid < float(params["shape_low_solidity_thr"])))
        low_circularity_frac = float(np.nanmean(circ < float(params["shape_low_circularity_thr"])))
        high_ecc_frac = float(np.nanmean(ecc > float(params["shape_high_ecc_thr"])))
    else:
        area_med = np.nan
        solidity_med = np.nan
        circularity_med = np.nan
        low_solidity_frac = np.nan
        low_circularity_frac = np.nan
        high_ecc_frac = np.nan

    return {
        "p_mass_capture": p_mass_capture,
        "high_recall": high_recall,
        "low_contam": low_contam,
        "n_obj": n_obj,
        "area_median": area_med,
        "solidity_median": solidity_med,
        "circularity_median": circularity_med,
        "low_solidity_frac": low_solidity_frac,
        "low_circularity_frac": low_circularity_frac,
        "high_ecc_frac": high_ecc_frac,
    }


def run_segmentation_scan(tag: str, seg_params: dict, compact_cfg: dict | None = None):
    frame_rows = []
    obj_rows = []
    t0 = time.time()
    for t in range(N_TIME):
        p_nuc = prob_mm[t, 0].astype(np.float32, copy=False)
        seg = segment_probability_frame(
            p_nuc=p_nuc,
            valid_mask=valid_mask,
            params=seg_params,
            compact_cfg=compact_cfg,
            return_debug=False,
        )
        qm = frame_quality_metrics(p_nuc=p_nuc, labels=seg["labels"], valid_mask=valid_mask, params=seg_params)
        qm.update(
            {
                "scan_tag": tag,
                "t": int(t),
                "time_hr": float(t * MIN_PER_FRAME / 60.0),
            }
        )
        frame_rows.append(qm)

        rdf = seg["regions_df"].copy()
        if not rdf.empty:
            rdf["scan_tag"] = tag
            rdf["t"] = int(t)
            rdf["time_hr"] = float(t * MIN_PER_FRAME / 60.0)
            rdf["shape_score"] = (
                rdf["solidity"].fillna(0.0)
                + rdf["circularity"].fillna(0.0)
                - 0.20 * rdf["eccentricity"].fillna(0.0)
            )
            obj_rows.append(rdf)

    frame_df = pd.DataFrame(frame_rows)
    obj_df = pd.concat(obj_rows, ignore_index=True) if obj_rows else pd.DataFrame()
    frame_path = TABLE_DIR / f"{tag}_frame_metrics.csv"
    obj_path = TABLE_DIR / f"{tag}_object_metrics.csv"
    frame_df.to_csv(frame_path, index=False)
    obj_df.to_csv(obj_path, index=False)
    elapsed = time.time() - t0
    print(f"{tag}: scan completed in {elapsed:.1f}s")
    print(f"Saved: {frame_path}")
    print(f"Saved: {obj_path}")
    return frame_df, obj_df


def fetch_frame_arrays(t: int):
    bf = raw_mm[t, 0].astype(np.float32, copy=False)
    rfp = raw_mm[t, 1].astype(np.float32, copy=False)
    p_nuc = prob_mm[t, 0].astype(np.float32, copy=False)
    return bf, rfp, p_nuc


def fetch_cropped_views(t: int, cy: float, cx: float, labels: np.ndarray, half_size: int = FAILURE_CROP_HALF_SIZE):
    bf, rfp, p_nuc = fetch_frame_arrays(t)
    ys, xs = crop_slices(cy, cx, half_size=half_size, shape=bf.shape)
    return {
        "bf": bf[ys, xs],
        "rfp": rfp[ys, xs],
        "p_nuc": p_nuc[ys, xs],
        "labels": labels[ys, xs],
        "valid": valid_mask[ys, xs],
        "ys": ys,
        "xs": xs,
    }


## 4) Step-By-Step Baseline Diagnostics

Before scanning the whole time series, it helps to look at a few frames in detail and make sure the current
baseline is doing what we think it is doing.

This section shows the **segmentation pipeline only**. It does not include tracking and it does not include
fluorescence quantification.

The key stages to compare are:

- `watershed seeds`: candidate nucleus centers (local maxima in `p_nuc`) overlaid on RFP
- `post split-merge`: instance masks after watershed and merge-back, before final filtering
- `final boundaries`: masks after the final area / solidity / eccentricity / mean-probability filters

Because the TrackMate handoff will be easier to repair when recall is high, treat these audits with a
**false-negative-sensitive** mindset: missing real cells is usually more costly than carrying some extra debris.


In [ ]:
default_step_times = [0, N_TIME // 2, N_TIME - 1]
step_times = STEP_DIAG_TIMES if STEP_DIAG_TIMES is not None else default_step_times
step_times = sorted({int(t) for t in step_times if 0 <= int(t) < N_TIME})
print("Step diagnostic times:", step_times)

for t in step_times:
    bf, rfp, p_nuc = fetch_frame_arrays(t)
    seg_dbg = segment_probability_frame(
        p_nuc=p_nuc,
        valid_mask=valid_mask,
        params=BASELINE_SEG_PARAMS,
        compact_cfg=None,
        return_debug=True,
    )
    dbg = seg_dbg["debug_steps"]

    fig, ax = plt.subplots(3, 4, figsize=(16, 10), constrained_layout=True)
    ax = ax.ravel()

    panels = [
        ("BF raw", _robust_gray_display(bf)),
        ("RFP raw", _robust_gray_display(rfp)),
        ("p_nuc raw", dbg["p_raw"]),
        ("p_nuc smooth", dbg["p_smooth"]),
        ("low threshold", dbg["mask_low"]),
        ("high threshold", dbg["mask_high"]),
        ("hysteresis", dbg["hys_after_fallback"]),
        ("post morph", dbg["hys_post_morph"]),
        ("watershed seeds on RFP", overlay_seed_points(rfp, seg_dbg["peak_coords"], mask=valid_mask)),
        ("post split-merge on RFP", overlay_boundaries(rfp, dbg["labels_splitmerge"], mask=valid_mask)),
        ("final boundaries on RFP", overlay_boundaries(rfp, dbg["labels_final"], mask=valid_mask)),
        ("final boundaries on BF", overlay_boundaries(bf, dbg["labels_final"], mask=valid_mask)),
    ]

    for axis, (title, img) in zip(ax, panels):
        if img.ndim == 2:
            cmap = "gray" if title in {"BF raw", "RFP raw"} else None
            axis.imshow(img, cmap=cmap)
        else:
            axis.imshow(img)
        axis.set_title(title)
        axis.set_xticks([])
        axis.set_yticks([])

    fig.suptitle(f"{POSITION_LABEL} baseline stepwise diagnostics, t={t}", fontsize=14)
    out = QUICK_DIR / f"baseline_stepwise_t{t:03d}.png"
    fig.savefig(out, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", out)


## 5) Consecutive-Frame Baseline Audit

Single frames are not always enough to decide whether a feature is a real cell, dead-cell debris, or a
transient aggregate. These short windows show consecutive frames around the representative audit times so
you can judge temporal stability and whether we are missing cells versus overcalling debris.


In [ ]:
for center_t in step_times:
    t_start = max(0, int(center_t) - int(CONSECUTIVE_AUDIT_HALF_WINDOW))
    t_stop = min(N_TIME - 1, int(center_t) + int(CONSECUTIVE_AUDIT_HALF_WINDOW))
    window_times = list(range(t_start, t_stop + 1))
    print(f"Consecutive audit centered at t={center_t}: {window_times}")

    fig, axes = plt.subplots(len(window_times), 2, figsize=(10, 3.2 * len(window_times)), constrained_layout=True)
    if len(window_times) == 1:
        axes = np.array([axes])

    rows = []
    for i, t in enumerate(window_times):
        bf, rfp, p_nuc = fetch_frame_arrays(t)
        seg = segment_probability_frame(
            p_nuc=p_nuc,
            valid_mask=valid_mask,
            params=BASELINE_SEG_PARAMS,
            compact_cfg=None,
            return_debug=False,
        )
        n_obj = int(seg["labels"].max())
        rows.append({"t": int(t), "time_hr": float(t * MIN_PER_FRAME / 60.0), "n_obj": n_obj})

        axes[i, 0].imshow(overlay_boundaries(rfp, seg["labels"], mask=valid_mask))
        axes[i, 0].set_title(f"RFP boundaries | t={t} | n={n_obj}")
        axes[i, 1].imshow(overlay_boundaries(bf, seg["labels"], mask=valid_mask))
        axes[i, 1].set_title(f"BF boundaries | t={t} | n={n_obj}")
        for j in range(2):
            axes[i, j].set_xticks([])
            axes[i, j].set_yticks([])

    fig.suptitle(
        f"{POSITION_LABEL} consecutive baseline audit around t={center_t} "
        f"(TrackMate-oriented: prefer some extras over missing cells)",
        fontsize=13,
    )
    out = QUICK_DIR / f"baseline_consecutive_audit_center_t{center_t:03d}.png"
    fig.savefig(out, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", out)
    display(pd.DataFrame(rows))


## 6) Full Baseline Segmentation Scan

This pass computes frame-level and object-level tables for the whole movie. Those tables become the basis for:

- time trends
- suspicious-object mining
- good-vs-bad galleries
- refinement comparison


In [ ]:
if RUN_FULL_BASELINE_SCAN:
    baseline_frame_df, baseline_obj_df = run_segmentation_scan(
        tag="baseline",
        seg_params=BASELINE_SEG_PARAMS,
        compact_cfg=None,
    )
else:
    baseline_frame_df = pd.read_csv(TABLE_DIR / "baseline_frame_metrics.csv")
    baseline_obj_df = pd.read_csv(TABLE_DIR / "baseline_object_metrics.csv")

print("Baseline frames:", baseline_frame_df.shape)
print("Baseline objects:", baseline_obj_df.shape)
display(baseline_frame_df.head())
display(baseline_obj_df.head())


## 7) Baseline Trends Over Time

These summaries are not the decision-maker on their own, but they help us find where the segmentation is
becoming more irregular over the course of the movie.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
time_hr = baseline_frame_df["time_hr"].to_numpy()

axes[0, 0].plot(time_hr, baseline_frame_df["n_obj"], color="tab:red", lw=1.8)
axes[0, 0].set_title("Object count")
axes[0, 0].set_xlabel("Time (hours)")
axes[0, 0].set_ylabel("objects / frame")

axes[0, 1].plot(time_hr, baseline_frame_df["p_mass_capture"], color="tab:blue", lw=1.8)
axes[0, 1].set_title("Probability mass captured")
axes[0, 1].set_xlabel("Time (hours)")
axes[0, 1].set_ylabel("fraction")

axes[1, 0].plot(time_hr, baseline_frame_df["low_solidity_frac"], label="low solidity", lw=1.8)
axes[1, 0].plot(time_hr, baseline_frame_df["low_circularity_frac"], label="low circularity", lw=1.8)
axes[1, 0].plot(time_hr, baseline_frame_df["high_ecc_frac"], label="high eccentricity", lw=1.8)
axes[1, 0].legend()
axes[1, 0].set_title("Suspicious-shape fractions")
axes[1, 0].set_xlabel("Time (hours)")
axes[1, 0].set_ylabel("fraction of objects")

axes[1, 1].plot(time_hr, baseline_frame_df["solidity_median"], label="median solidity", lw=1.8)
axes[1, 1].plot(time_hr, baseline_frame_df["circularity_median"], label="median circularity", lw=1.8)
axes[1, 1].legend()
axes[1, 1].set_title("Median shape metrics")
axes[1, 1].set_xlabel("Time (hours)")
axes[1, 1].set_ylabel("value")

trend_path = QUICK_DIR / "baseline_time_trends.png"
fig.savefig(trend_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved:", trend_path)

display(
    baseline_frame_df[
        [
            "n_obj",
            "p_mass_capture",
            "solidity_median",
            "circularity_median",
            "low_solidity_frac",
            "low_circularity_frac",
            "high_ecc_frac",
        ]
    ].describe()
)


## 8) Failure Gallery From The Current Baseline

This is the main “where should we improve next?” section.

The gallery is built from full-movie object metrics and grouped into:

- low solidity
- low circularity
- high eccentricity
- high-confidence good examples

The intent is to make it obvious whether the problem looks like:

- a classifier problem (`p_nuc` is already confused)
- or a post-processing problem (good `p_nuc`, bad final contour)


In [ ]:
def pick_gallery_rows(obj_df: pd.DataFrame, mode: str, n: int) -> pd.DataFrame:
    if obj_df.empty:
        return obj_df.copy()
    if mode == "low_solidity":
        df = obj_df.sort_values(["solidity", "area"], ascending=[True, False])
    elif mode == "low_circularity":
        df = obj_df.sort_values(["circularity", "area"], ascending=[True, False])
    elif mode == "high_eccentricity":
        df = obj_df.sort_values(["eccentricity", "area"], ascending=[False, False])
    elif mode == "good":
        df = obj_df.copy()
        df["good_score"] = (
            df["shape_score"].fillna(-np.inf)
            + 0.20 * df["mean_prob"].fillna(0.0)
            + 0.002 * np.clip(df["area"].fillna(0.0), 0, 400)
        )
        df = df.sort_values(["good_score"], ascending=[False])
    else:
        raise ValueError(f"Unknown gallery mode: {mode}")

    chosen = []
    used = set()
    for _, row in df.iterrows():
        key = (int(row["t"]), int(round(row["centroid_y"] // 16)), int(round(row["centroid_x"] // 16)))
        if key in used:
            continue
        used.add(key)
        chosen.append(row)
        if len(chosen) >= n:
            break
    return pd.DataFrame(chosen).reset_index(drop=True)


def render_gallery(rows_df: pd.DataFrame, title: str, out_path: Path):
    if rows_df.empty:
        print(f"No rows to render for {title}")
        return

    n = len(rows_df)
    fig, axes = plt.subplots(n, 4, figsize=(12, 3.2 * n), constrained_layout=True)
    if n == 1:
        axes = np.array([axes])

    for i, (_, row) in enumerate(rows_df.iterrows()):
        t = int(row["t"])
        cy = float(row["centroid_y"])
        cx = float(row["centroid_x"])
        seg = segment_probability_frame(
            p_nuc=prob_mm[t, 0].astype(np.float32, copy=False),
            valid_mask=valid_mask,
            params=BASELINE_SEG_PARAMS,
            compact_cfg=None,
            return_debug=False,
        )
        crop = fetch_cropped_views(t=t, cy=cy, cx=cx, labels=seg["labels"], half_size=FAILURE_CROP_HALF_SIZE)

        axes[i, 0].imshow(_robust_gray_display(crop["bf"], mask=crop["valid"]), cmap="gray")
        axes[i, 0].set_title(f"BF\nt={t}")

        axes[i, 1].imshow(_robust_gray_display(crop["rfp"], mask=crop["valid"]), cmap="gray")
        axes[i, 1].set_title(
            "RFP\n"
            f"area={int(row['area'])}, mean_p={row['mean_prob']:.2f}"
        )

        axes[i, 2].imshow(crop["p_nuc"], cmap="viridis", vmin=0, vmax=1)
        axes[i, 2].set_title(
            "p_nuc\n"
            f"sol={row['solidity']:.2f}, circ={row['circularity']:.2f}"
        )

        axes[i, 3].imshow(overlay_boundaries(crop["rfp"], crop["labels"], mask=crop["valid"]))
        axes[i, 3].set_title(f"Overlay\necc={row['eccentricity']:.2f}")

        for j in range(4):
            axes[i, j].set_xticks([])
            axes[i, j].set_yticks([])

    fig.suptitle(title, fontsize=14)
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", out_path)


low_sol_gallery = pick_gallery_rows(baseline_obj_df, mode="low_solidity", n=FAILURE_GALLERY_N)
low_circ_gallery = pick_gallery_rows(baseline_obj_df, mode="low_circularity", n=FAILURE_GALLERY_N)
high_ecc_gallery = pick_gallery_rows(baseline_obj_df, mode="high_eccentricity", n=FAILURE_GALLERY_N)
good_gallery = pick_gallery_rows(baseline_obj_df, mode="good", n=GOOD_GALLERY_N)

display(low_sol_gallery[["t", "area", "solidity", "circularity", "eccentricity", "mean_prob"]])
display(low_circ_gallery[["t", "area", "solidity", "circularity", "eccentricity", "mean_prob"]])
display(high_ecc_gallery[["t", "area", "solidity", "circularity", "eccentricity", "mean_prob"]])
display(good_gallery[["t", "area", "solidity", "circularity", "eccentricity", "mean_prob"]])

render_gallery(
    low_sol_gallery,
    title="Baseline failure gallery: low-solidity objects",
    out_path=GALLERY_DIR / "baseline_gallery_low_solidity.png",
)
render_gallery(
    low_circ_gallery,
    title="Baseline failure gallery: low-circularity objects",
    out_path=GALLERY_DIR / "baseline_gallery_low_circularity.png",
)
render_gallery(
    high_ecc_gallery,
    title="Baseline failure gallery: high-eccentricity objects",
    out_path=GALLERY_DIR / "baseline_gallery_high_eccentricity.png",
)
render_gallery(
    good_gallery,
    title="Baseline gallery: high-confidence compact objects",
    out_path=GALLERY_DIR / "baseline_gallery_good_objects.png",
)


## 9) First Refinement Candidate: Compactify Ragged Masks

This is intentionally modest. We are **not** retraining ilastik here. We are testing whether a guarded
post-segmentation compactification step improves the worst masks without obviously breaking the good ones.


In [ ]:
if RUN_REFINEMENT_SCAN:
    refined_frame_df, refined_obj_df = run_segmentation_scan(
        tag="compact_refine",
        seg_params=BASELINE_SEG_PARAMS,
        compact_cfg=REFINE_CFG,
    )
else:
    refined_frame_df = pd.read_csv(TABLE_DIR / "compact_refine_frame_metrics.csv")
    refined_obj_df = pd.read_csv(TABLE_DIR / "compact_refine_object_metrics.csv")

compare_summary = pd.DataFrame(
    [
        {
            "metric": "median objects/frame",
            "baseline": float(np.nanmedian(baseline_frame_df["n_obj"])),
            "compact_refine": float(np.nanmedian(refined_frame_df["n_obj"])),
        },
        {
            "metric": "median p_mass_capture",
            "baseline": float(np.nanmedian(baseline_frame_df["p_mass_capture"])),
            "compact_refine": float(np.nanmedian(refined_frame_df["p_mass_capture"])),
        },
        {
            "metric": "median solidity",
            "baseline": float(np.nanmedian(baseline_frame_df["solidity_median"])),
            "compact_refine": float(np.nanmedian(refined_frame_df["solidity_median"])),
        },
        {
            "metric": "median circularity",
            "baseline": float(np.nanmedian(baseline_frame_df["circularity_median"])),
            "compact_refine": float(np.nanmedian(refined_frame_df["circularity_median"])),
        },
        {
            "metric": "median low_solidity_frac",
            "baseline": float(np.nanmedian(baseline_frame_df["low_solidity_frac"])),
            "compact_refine": float(np.nanmedian(refined_frame_df["low_solidity_frac"])),
        },
        {
            "metric": "median low_circularity_frac",
            "baseline": float(np.nanmedian(baseline_frame_df["low_circularity_frac"])),
            "compact_refine": float(np.nanmedian(refined_frame_df["low_circularity_frac"])),
        },
    ]
)
display(compare_summary)
compare_summary.to_csv(TABLE_DIR / "baseline_vs_compact_refine_summary.csv", index=False)


In [ ]:
compare_times = sorted({0, 20, 80, 120, 160, 220, 280, N_TIME - 1})
fig, axes = plt.subplots(len(compare_times), 3, figsize=(11, 3.4 * len(compare_times)), constrained_layout=True)
if len(compare_times) == 1:
    axes = np.array([axes])

for i, t in enumerate(compare_times):
    bf, rfp, p_nuc = fetch_frame_arrays(t)
    base_seg = segment_probability_frame(
        p_nuc=p_nuc,
        valid_mask=valid_mask,
        params=BASELINE_SEG_PARAMS,
        compact_cfg=None,
    )
    ref_seg = segment_probability_frame(
        p_nuc=p_nuc,
        valid_mask=valid_mask,
        params=BASELINE_SEG_PARAMS,
        compact_cfg=REFINE_CFG,
    )

    axes[i, 0].imshow(_robust_gray_display(rfp), cmap="gray")
    axes[i, 0].set_title(f"RFP raw\nt={t}")
    axes[i, 1].imshow(overlay_boundaries(rfp, base_seg["labels"]))
    axes[i, 1].set_title(f"Baseline\nn={int(base_seg['labels'].max())}")
    axes[i, 2].imshow(overlay_boundaries(rfp, ref_seg["labels"], color_rgb=(0.2, 1.0, 0.2)))
    axes[i, 2].set_title(f"Compact refine\nn={int(ref_seg['labels'].max())}")
    for j in range(3):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

compare_path = QUICK_DIR / "baseline_vs_compact_refine_fullframe.png"
fig.savefig(compare_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved:", compare_path)


In [ ]:
suspicious_union = pd.concat(
    [
        low_sol_gallery.assign(source="low_solidity"),
        low_circ_gallery.assign(source="low_circularity"),
        high_ecc_gallery.assign(source="high_eccentricity"),
    ],
    ignore_index=True,
)
suspicious_union = suspicious_union.drop_duplicates(subset=["t", "centroid_y", "centroid_x"]).reset_index(drop=True)

n = min(len(suspicious_union), FAILURE_GALLERY_N)
if n > 0:
    fig, axes = plt.subplots(n, 5, figsize=(15, 3.4 * n), constrained_layout=True)
    if n == 1:
        axes = np.array([axes])

    for i in range(n):
        row = suspicious_union.iloc[i]
        t = int(row["t"])
        cy = float(row["centroid_y"])
        cx = float(row["centroid_x"])
        bf, rfp, p_nuc = fetch_frame_arrays(t)
        base_seg = segment_probability_frame(
            p_nuc=p_nuc,
            valid_mask=valid_mask,
            params=BASELINE_SEG_PARAMS,
            compact_cfg=None,
        )
        ref_seg = segment_probability_frame(
            p_nuc=p_nuc,
            valid_mask=valid_mask,
            params=BASELINE_SEG_PARAMS,
            compact_cfg=REFINE_CFG,
        )
        crop_base = fetch_cropped_views(t=t, cy=cy, cx=cx, labels=base_seg["labels"])
        crop_ref = fetch_cropped_views(t=t, cy=cy, cx=cx, labels=ref_seg["labels"])

        axes[i, 0].imshow(_robust_gray_display(crop_base["bf"], mask=crop_base["valid"]), cmap="gray")
        axes[i, 0].set_title(f"BF\n{row['source']}, t={t}")
        axes[i, 1].imshow(_robust_gray_display(crop_base["rfp"], mask=crop_base["valid"]), cmap="gray")
        axes[i, 1].set_title("RFP")
        axes[i, 2].imshow(crop_base["p_nuc"], cmap="viridis", vmin=0, vmax=1)
        axes[i, 2].set_title("p_nuc")
        axes[i, 3].imshow(overlay_boundaries(crop_base["rfp"], crop_base["labels"], mask=crop_base["valid"]))
        axes[i, 3].set_title("Baseline")
        axes[i, 4].imshow(
            overlay_boundaries(
                crop_ref["rfp"],
                crop_ref["labels"],
                color_rgb=(0.2, 1.0, 0.2),
                mask=crop_ref["valid"],
            )
        )
        axes[i, 4].set_title("Compact refine")

        for j in range(5):
            axes[i, j].set_xticks([])
            axes[i, j].set_yticks([])

    suspicious_compare_path = QUICK_DIR / "suspicious_objects_baseline_vs_compact_refine.png"
    fig.savefig(suspicious_compare_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", suspicious_compare_path)
else:
    print("No suspicious objects found to compare.")


## 10) How To Read This Notebook

The practical decision loop is:

1. Look at the **failure galleries** first.
2. Ask whether the probability map itself is confused, or whether the final contour is the main problem.
3. Check whether the compact-refine comparison helps the obvious half-moon / concave cases.
4. If the probability map is the limiting factor, do another focused ilastik training round on those hard examples.
5. Only after the segmentation looks better should we regenerate the Fiji preliminary tracks.

In other words:

- if `p_nuc` already looks right and the contour looks wrong -> fix postprocessing here
- if `p_nuc` already confuses blebs and nuclei -> retrain ilastik
